# No-RAG Baseline

In [24]:
# === Imports & Config ===
import os, json, re, math, pathlib, itertools, collections,getpass
import pandas as pd
import numpy as np
from openai import OpenAI
import matplotlib.pyplot as plt
import fitz

CODEBOOK_MD = "/Users/xinby/Desktop/AI44PT/data/processed/TheCodingTask.md"
PAPER_PDF = "/Users/xinby/Desktop/AI44PT/data/processed/sample_paper.pdf"
CLS_MODEL = "gpt-4o-mini"  

# Batch assessment settings
MAX_ITEMS = 1  # Number of items to process at a time (OpenAI API limit), None for all at once
SEED = 7          # Random seed
if not os.environ.get('OPENAI_API_KEY'):
  os.environ["OPENAI_API_KEY"] = getpass.getpass("🔑 Enter OPENAI_API_KEY: ")
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
print("OpenAI client ok. Using model:", CLS_MODEL)

CODEBOOK_TEXT = pathlib.Path(CODEBOOK_MD).read_text(encoding="utf-8")
print("Loaded Codebook.md chars:", len(CODEBOOK_TEXT))


OpenAI client ok. Using model: gpt-4o-mini
Loaded Codebook.md chars: 43556


In [ ]:
CODE_MODE = "local"  # "colab", "local", or "manual_upload"

if CODE_MODE == "colab":
    from google.colab import drive
    drive.mount('/content/drive')
    PAPER_PDF = "/content/drive/MyDrive/AI44PT/data/processed/sample_paper.pdf"
    # CODEBOOK_PDF = "/content/drive/MyDrive/AI44PT/data/processed/4ptCodebook.pdf"
    CODEBOOK_MD = "/content/drive/MyDrive/AI44PT/data/processed/TheCodingTask.md"
elif CODE_MODE == "local":
    import os
    import pathlib
    cwd = os.getcwd()
    cwd_path = pathlib.Path(cwd)
    cwd = str(cwd_path.parent)
    print(f"Current working directory: {cwd}")
    PAPER_PDF = os.path.join(cwd, "data/processed/sample_paper.pdf")
    # CODEBOOK_PDF = os.path.join(cwd, "data/processed/4ptCodebook.pdf")
    CODEBOOK_MD = os.path.join(cwd, "data/processed/TheCodingTask.md")
elif CODE_MODE == "manual_upload":
    from google.colab import files
    print('Upload your PDFs. Please name the codebook file "TheCodingTask.md" and the paper file "sample_paper.pdf".')
    uploaded = files.upload()  # choose files in the UI
    print('Files uploaded:', list(uploaded.keys()))
    CODEBOOK_FILENAME = "TheCodingTask.md"
    DOC_FILENAME = "sample_paper.pdf"

    if CODEBOOK_FILENAME in uploaded and DOC_FILENAME in uploaded:
        CODEBOOK_MD = CODEBOOK_FILENAME
        PAPER_PDF = DOC_FILENAME
        print(f"Codebook file: {CODEBOOK_MD}, Document file: {PAPER_PDF}")
    else:
        print(f"Error: Please ensure you upload both '{CODEBOOK_FILENAME}' and '{DOC_FILENAME}'.")
        CODEBOOK_MD = None
        PAPER_PDF = None

In [18]:
def read_pdf(path: str):
    """
    Read a PDF file and extract text from each page.
    Returns a list of dictionaries with 'page' and 'text' keys.
    """
    doc = fitz.open(path)
    pages = []
    for i, pg in enumerate(doc):
        pages.append({'page': i+1, 'text': pg.get_text('text') or ''})
    return pages

def read_markdown(path: str, heading_pattern: str = r'^#{1,6}\s'):
    """
    Read a markdown file and split it into sections based on headings.
    Each section starts with a heading matching the given pattern.
    Returns a list of dictionaries with 'page' (section index) and 'text'.
    """
    heading_re = re.compile(heading_pattern)
    sections = []
    current_lines = []
    section_idx = 1
    with open(path, 'r', encoding='utf-8') as f:
        for raw_line in f:
            line = raw_line.rstrip('\n')
            if heading_re.match(line.strip()):
                if current_lines:
                    sections.append({'page': section_idx, 'text': '\n'.join(current_lines).strip()})
                    section_idx += 1
                current_lines = [line]
            else:
                current_lines.append(line)
        if current_lines:
            sections.append({'page': section_idx, 'text': '\n'.join(current_lines).strip()})
    if not sections:
        text = pathlib.Path(path).read_text(encoding='utf-8')
        sections.append({'page': 1, 'text': text})
    return sections

cb_pages = read_markdown(CODEBOOK_MD)
paper_pages = read_pdf(PAPER_PDF)
print("Codebook sections:", len(cb_pages), "Sample:")
print(*cb_pages[:1], sep="\n")
print("Paper pages:", len(paper_pages), "Sample:")
print(*paper_pages[:1], sep="\n")

Codebook sections: 13 Sample:
{'page': 1, 'text': '# The Coding Task\n\n***This codebook develops a systematic and standardized way to classify qualitative documents into one of Cashore’s four problem types(4PT) framework*** . It will do so by providing guidance to coders on how to assign Type 1, 2, 3 or 4 status to qualitative text including, but not limited to, peer reviewed journal articles, course syllabi, policy statements, policy analyses, media communications and policy recommendations. This classification effort complements, rather than replaces, careful qualitative application\nof the framework required for uncovering important nuances through deliberative analysis and reflection of highly complex text. ***What the quantitative coding effort does well is to generate useful knowledge about whether the overall patterns identified by an one study or set of studies compare to other cases outside of the study. Coding also permits assessments as to whether there are differences in t

In [30]:
# === JSON Schema for 4PT Analysis ===
FOURPT_ANALYSIS_SCHEMA = {
    "name": "FourPTAnalysis",
    "schema": {
        "type": "object",
        "properties": {
            "Q1_sustainability_fit": {
                "type": "string",
                "enum": ["Yes", "No"],
                "description": "Does the article fit in the universe of sustainability analyses we seek to assess?"
            },
            "Q2_problems_addressed": {
                "type": "string",
                "description": "What problems or set of problems is the article trying to address?",
                "maxLength": 500
            },
            "Q3_on_ground_problem": {
                "type": "string",
                "enum": ["Yes", "No"],
                "description": "Derived from understanding/managing clearly specified on-ground problems?"
            },
            "Q4_on_ground_arguments": {
                "type": "string",
                "description": "Arguments supporting Q3 response with key text citations",
                "maxLength": 800
            },
            "Q5_beyond_on_ground": {
                "type": "string",
                "enum": ["Yes", "No"],
                "description": "Generated to apply beyond clearly specified on-ground problems?"
            },
            "Q6_beyond_arguments": {
                "type": "string",
                "description": "Arguments supporting Q5 response with key text citations",
                "maxLength": 800
            },
            "Q7_utility_maximization": {
                "type": "string",
                "enum": ["Yes", "No"],
                "description": "Treat entities as self-interested, utility-maximizing agents?"
            },
            "Q8_utility_arguments": {
                "type": "string",
                "description": "Arguments supporting Q7 response with key text citations",
                "maxLength": 800
            },
            "Q9_beyond_self_interest": {
                "type": "string",
                "enum": ["Yes", "No"],
                "description": "Extends beyond self-interested satisfaction seeking motivations?"
            },
            "Q10_beyond_self_interest_arguments": {
                "type": "string",
                "description": "Arguments supporting Q9 response with key text citations",
                "maxLength": 800
            },
            "Q11_fourpt_type": {
                "type": "string",
                "enum": ["T1", "T2", "T3", "T4"],
                "description": "Final 4PT Type classification"
            },
            "Q11_justification": {
                "type": "string",
                "description": "Justification for 4PT type (≤120 characters)",
                "maxLength": 120
            },
            "Q12_whack_a_mole_risk": {
                "type": "string",
                "enum": ["Yes", "No"],
                "description": "Detect whack-a-mole risk or solving T4 with T1/2/3 thinking?"
            },
            "Q12_evidence_spans": {
                "type": "array",
                "items": {"type": "string", "maxLength": 300},
                "maxItems": 2,
                "minItems": 0,
                "description": "0-2 evidence spans for whack-a-mole risk"
            }
        },
        "required": [
            "Q1_sustainability_fit",
            "Q2_problems_addressed", 
            "Q3_on_ground_problem",
            "Q4_on_ground_arguments",
            "Q5_beyond_on_ground",
            "Q6_beyond_arguments",
            "Q7_utility_maximization",
            "Q8_utility_arguments",
            "Q9_beyond_self_interest",
            "Q10_beyond_self_interest_arguments",
            "Q11_fourpt_type",
            "Q11_justification",
            "Q12_whack_a_mole_risk",
            "Q12_evidence_spans"
        ],
        "additionalProperties": False
    },
    "strict": True
}

In [31]:
def analyze_article_fourpt(article_pages: list, codebook_pages: list, model: str = CLS_MODEL):
    """
    使用4PT框架分析文章，一次性回答所有12个问题
    """
    # 将页面内容合并为文本
    article_text = "\n\n".join([f"Page {p['page']}:\n{p['text']}" for p in article_pages])
    codebook_text = "\n\n".join([f"Section {p['page']}:\n{p['text']}" for p in codebook_pages])
    
    prompt = f"""
You are an expert public policy analyst reviewing sustainability research articles. 

**Instructions:**
- Answer ALL questions based on the provided Codebook and Article
- Provide specific citations when requested
- Keep justifications concise and evidence-based
- For Yes/No questions, choose definitively based on evidence

**4PT Codebook:**
{codebook_text}

**Article to Analyze:**
{article_text}

Please analyze this article systematically through the 4PT framework, answering each question thoroughly.
    """.strip()
    
    try:
        resp = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            response_format={"type": "json_schema", "json_schema": FOURPT_ANALYSIS_SCHEMA},
            max_tokens=2000,
            temperature=0.1
        )
        
        result = json.loads(resp.choices[0].message.content)
        
        # 添加一些派生字段便于分析
        result["analysis_metadata"] = {
            "model": model,
            "timestamp": pd.Timestamp.now().isoformat(),
            "on_ground_focus": result["Q3_on_ground_problem"] == "Yes",
            "beyond_application": result["Q5_beyond_on_ground"] == "Yes", 
            "utility_based": result["Q7_utility_maximization"] == "Yes",
            "beyond_self_interest": result["Q9_beyond_self_interest"] == "Yes"
        }
        
        return result
        
    except Exception as e:
        print(f"Error in analysis: {e}")
        return None

In [32]:
result = analyze_article_fourpt(paper_pages, cb_pages)

# 查看结果
if result:
    print(f"4PT Type: {result['Q11_fourpt_type']}")
    print(f"Justification: {result['Q11_justification']}")
    print(f"Whack-a-mole Risk: {result['Q12_whack_a_mole_risk']}")

4PT Type: T1
Justification: The article emphasizes utility-maximizing outcomes through problem-solving approaches focused on specific voluntary low碳
Whack-a-mole Risk: No


In [34]:
print(*result.items(), sep="\n")

('Q1_sustainability_fit', 'Yes')
('Q2_problems_addressed', 'The article addresses the effectiveness of voluntary programs in regulating and governing societal problems related to low carbon building and city development, focusing on how design conditions and diffusion networks affect program performance.')
('Q3_on_ground_problem', 'Yes')
('Q4_on_ground_arguments', 'The article examines 26 voluntary programs for low carbon building and city development, analyzing how specific conditions affect their performance, indicating a clear focus on specified on-ground problems (e.g., carbon emissions reduction).')
('Q5_beyond_on_ground', 'Yes')
('Q6_beyond_arguments', 'The findings suggest that the diffusion network perspective can be applied to understand voluntary program performance in various contexts, indicating its relevance beyond the specific cases studied (e.g., implications for other sectors and types of voluntary programs).')
('Q7_utility_maximization', 'Yes')
('Q8_utility_arguments',

For reference, the human coder's answer for this paper is:
1. Yes
2. Interaction of program design conditions in affecting performance of voluntary programs
3. No
4. Geographical focus in several countries with no clearly specified on the ground problem
5. Looks at the topic of low carbon building and city development across cases in Australia, the Netherlands and the US
6. Yes
7. Explores implications on voluntary program performance of the diffusion network
8. This article has introduced the diffusion network perspective as a critical condition to understand voluntary program performance
9. Analysis on generalizability is valid. Proceed to the next questions on utility.
10. Yes
11. Key objective: determinants of voluntary program performance, which originates from high cost of direct regulatory intervention
12. Voluntary programs aim to change the behavior of individuals and organizations, but without the force of law. They have rapidly become popular governance instruments in situations in which it is too costly or difficult to implement direct regulatory interventions, for instance because of political unwillingness (Darnall & Carmin 2005). They also provide an opportunity to showcase and market desired “beyond compliance” behavior, or to reward leading firms (Saurwein 2011).
13. No
14. Looks at how diffusion network perspective can relate to better voluntary program results
15. The diffusion network perspective, as conceptualized in this article, brings together these insights and argues that the stronger the diffusion network, the more likely it is that a voluntary program will achieve the desired results.
16. Analysis on generalizability is valid. Proceed to the next questions on utility.
17. Type 2
18. 4 - Hard